In [26]:
import pandas as pd
import re
from pathlib import Path

In [27]:
main_file = Path(r"D:\Proyek FEB\besrihin hasil scrap disini👌\checkpoint_1_merged_sitasi&Q_sinta.xlsx")
scimago_file = Path(r"D:\Proyek FEB\scimagojr 2025.csv")

df = pd.read_excel(main_file)

scimago = pd.read_csv(
    scimago_file,
    sep=";",
    quotechar='"',
    encoding="utf-8"
)

df.head()

,No,Title,Authors,Journal,SCOPUS Q,Tahun,Volume / Issue,LINK DOI/ARTIKEL,SCOPUS SITASI
0,1,Ethnic identity and internal migration decisio...,ILMIAWAN AUWALIN,Journal of Ethnic and Migration Studies,Q1,2019-01-11 00:00:00,"Vol. 14, Issue 13",10.1080/1369183X.2018.1561252,24
1,2,The role of service quality within Indonesian ...,BADRI MUNIR SUKOCO,Journal of Islamic Marketing,Q2,2020-01-14 00:00:00,"Vol. 11, Issue 1",10.1108/JIMA-03-2017-0033,60
2,3,Developing Islamic crowdfunding website platfo...,MUHAMAD NAFIK HADI RYANDONO,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48
3,4,Developing Islamic crowdfunding website platfo...,ACHSANIA HENDRATMI,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48
4,5,Developing Islamic crowdfunding website platfo...,PUJI SUCIA SUKMANINGRUM,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48


In [28]:
scimago.head()

,Rank,Sourceid,Title,Type,Issn,Publisher,Open Access,Open Access Diamond,SJR,SJR Best Quartile,...,Citations / Doc. (2years),Ref. / Doc.,%Female,Overton,Country,Region,Publisher.1,Coverage,Categories,Areas
0,1,28773,Ca-A Cancer Journal for Clinicians,journal,"15424863, 00079235",John Wiley and Sons Inc,No,No,"104,065",Q1,...,"285,55","90,23","46,34",2,United States,Northern America,John Wiley and Sons Inc,1950-2026,Hematology (Q1); Oncology (Q1),Medicine
1,2,20315,Nature Reviews Molecular Cell Biology,journal,"14710072, 14710080",Nature Research,No,No,"35,568",Q1,...,"45,22","98,42","34,87",0,United Kingdom,Western Europe,Nature Research,2000-2026,Cell Biology (Q1); Molecular Biology (Q1),"Biochemistry, Genetics and Molecular Biology"
2,3,29431,Quarterly Journal of Economics,journal,"00335533, 15314650",Oxford University Press,No,No,"33,069",Q1,...,"17,30","77,28","27,33",38,United Kingdom,Western Europe,Oxford University Press,1886-2026,Economics and Econometrics (Q1),"Economics, Econometrics and Finance"
3,4,20425,Nature Reviews Drug Discovery,journal,"14741784, 14741776",Nature Research,No,No,"32,577",Q1,...,"15,31","38,98","30,70",6,United Kingdom,Western Europe,Nature Research,2002-2026,Drug Discovery (Q1); Medicine (miscellaneous) ...,"Medicine; Pharmacology, Toxicology and Pharmac..."
4,5,17700156734,Nature Reviews Clinical Oncology,journal,"17594782, 17594774",Nature Publishing Group,No,No,"28,076",Q1,...,"36,54","82,40","30,71",2,United Kingdom,Western Europe,Nature Publishing Group,2009-2026,Oncology (Q1),Medicine


In [29]:
COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

SCIMAGO_TITLE = "Title"
SCIMAGO_Q = "SJR Best Quartile"
SCIMAGO_ISSN = "Issn"
SCIMAGO_CATEGORIES = "Categories"
SCIMAGO_COVERAGE = "Coverage"

In [30]:
def is_empty_value(value):
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    if text == "":
        return True
    
    if text.lower() in [
        "nan",
        "none",
        "null",
        "-",
        "?",
        "??",
        "???",
        "n/a",
        "na",
        "tidak ada",
        "not found"
    ]:
        return True
    
    return False


def normalize_journal(value):
    if pd.isna(value):
        return ""
    
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("&", "and")
    text = re.sub(r"[^a-z0-9\s]", "", text)
    
    return text.strip()


def clean_q(value):
    if is_empty_value(value):
        return None
    
    text = str(value).strip()
    
    match = re.search(r"\bQ\s*([1-4])\b", text, flags=re.IGNORECASE)
    
    if match:
        return f"Q{match.group(1)}"
    
    return None

In [31]:
scimago_clean = scimago.copy()

scimago_clean["journal_key"] = scimago_clean[SCIMAGO_TITLE].apply(normalize_journal)
scimago_clean["scimago_q_clean"] = scimago_clean[SCIMAGO_Q].apply(clean_q)

scimago_clean[
    [SCIMAGO_TITLE, SCIMAGO_Q, "scimago_q_clean", SCIMAGO_ISSN, SCIMAGO_COVERAGE]
].head()

,Title,SJR Best Quartile,scimago_q_clean,Issn,Coverage
0,Ca-A Cancer Journal for Clinicians,Q1,Q1,"15424863, 00079235",1950-2026
1,Nature Reviews Molecular Cell Biology,Q1,Q1,"14710072, 14710080",2000-2026
2,Quarterly Journal of Economics,Q1,Q1,"00335533, 15314650",1886-2026
3,Nature Reviews Drug Discovery,Q1,Q1,"14741784, 14741776",2002-2026
4,Nature Reviews Clinical Oncology,Q1,Q1,"17594782, 17594774",2009-2026


In [32]:
scimago_clean["scimago_q_clean"].value_counts(dropna=False)

scimago_q_clean
Q1      9470
Q2      7916
Q3      7116
Q4      6585
None    1106
Name: count, dtype: int64

In [33]:
rows_missing_q = df[df[COL_Q].apply(is_empty_value)].copy()

unique_journals = (
    rows_missing_q[[COL_JOURNAL]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

unique_journals["journal_key"] = unique_journals[COL_JOURNAL].apply(normalize_journal)

print("Jumlah baris Q kosong:", len(rows_missing_q))
print("Jumlah jurnal unik yang perlu dicocokkan:", len(unique_journals))

Jumlah baris Q kosong: 1524
Jumlah jurnal unik yang perlu dicocokkan: 389


In [34]:
scimago_map = (
    scimago_clean
    .dropna(subset=["journal_key"])
    .drop_duplicates(subset=["journal_key"], keep="first")
    .set_index("journal_key")
    .to_dict(orient="index")
)

In [35]:
match_results = []

for _, row in unique_journals.iterrows():
    journal = row[COL_JOURNAL]
    journal_key = row["journal_key"]
    
    if journal_key in scimago_map:
        item = scimago_map[journal_key]
        
        match_results.append({
            "journal": journal,
            "journal_key": journal_key,
            "matched_title": item.get(SCIMAGO_TITLE),
            "scimago_q": item.get("scimago_q_clean"),
            "issn": item.get(SCIMAGO_ISSN),
            "coverage": item.get(SCIMAGO_COVERAGE),
            "categories": item.get(SCIMAGO_CATEGORIES),
            "match_type": "exact_name",
            "status": "found"
        })
    else:
        match_results.append({
            "journal": journal,
            "journal_key": journal_key,
            "matched_title": None,
            "scimago_q": None,
            "issn": None,
            "coverage": None,
            "categories": None,
            "match_type": None,
            "status": "not_found"
        })

scimago_match_df = pd.DataFrame(match_results)

scimago_match_df.head()

,journal,journal_key,matched_title,scimago_q,issn,coverage,categories,match_type,status
0,Social Responsibility Journal,social responsibility journal,Social Responsibility Journal,Q1,"1758857X, 17471117",2005-2026,"Business, Management and Accounting (miscellan...",exact_name,found
1,Journal of Retailing and Consumer Services,journal of retailing and consumer services,Journal of Retailing and Consumer Services,Q1,09696989,1994-2026,Marketing (Q1),exact_name,found
2,Journal of Applied Accounting Research,journal of applied accounting research,Journal of Applied Accounting Research,Q1,09675426,"1999-2002, 2004-2006, 2008-2026","Accounting (Q1); Economics, Econometrics and F...",exact_name,found
3,Journal of Islamic Marketing,journal of islamic marketing,Journal of Islamic Marketing,Q2,"17590833, 17590841",2010-2026,Marketing (Q2),exact_name,found
4,Journal Data in Brief,journal data in brief,None,None,None,None,None,None,not_found


In [36]:
scimago_match_df["status"].value_counts(dropna=False)

status
found        302
not_found     87
Name: count, dtype: int64

In [37]:
scimago_match_df["scimago_q"].value_counts(dropna=False)

scimago_q
Q1      134
None     92
Q2       74
Q3       62
Q4       27
Name: count, dtype: int64

In [38]:
scimago_not_found = scimago_match_df[scimago_match_df["status"] != "found"].copy()

scimago_not_found.to_excel("checkpoint_2_scimago_not_found.xlsx", index=False)

len(scimago_not_found)

87

In [52]:
mask_found_but_q_empty = (
    (scimago_match_df["status"] == "found") &
    (scimago_match_df["scimago_q"].apply(is_empty_value))
)

print("Jumlah status found tapi Q kosong:", mask_found_but_q_empty.sum())

scimago_match_df.loc[mask_found_but_q_empty, "scimago_q"] = "no-Q"
scimago_match_df.loc[mask_found_but_q_empty, "status"] = "found_no_q"

Jumlah status found tapi Q kosong: 0


In [53]:
df_checkpoint2 = df.copy()

df_checkpoint2["journal_key"] = df_checkpoint2[COL_JOURNAL].apply(normalize_journal)

q_map = (
    scimago_match_df[scimago_match_df["status"] == "found"]
    .dropna(subset=["scimago_q"])
    .drop_duplicates(subset=["journal_key"])
    .set_index("journal_key")["scimago_q"]
    .to_dict()
)

In [54]:
updated_q_count = 0

for idx, row in df_checkpoint2.iterrows():
    journal_key = row["journal_key"]
    
    if journal_key not in q_map:
        continue
    
    if is_empty_value(row[COL_Q]):
        q_value = q_map[journal_key]
        
        if not is_empty_value(q_value):
            df_checkpoint2.at[idx, COL_Q] = q_value
            updated_q_count += 1

print("SCOPUS Q berhasil dilengkapi dari database SCImago:", updated_q_count)

SCOPUS Q berhasil dilengkapi dari database SCImago: 1079


In [55]:
df_checkpoint2 = df_checkpoint2.drop(columns=["journal_key"])

In [56]:
before_missing_q = df[COL_Q].apply(is_empty_value).sum()
after_missing_q = df_checkpoint2[COL_Q].apply(is_empty_value).sum()

comparison = pd.DataFrame({
    "metric": ["SCOPUS Q missing"],
    "before": [before_missing_q],
    "after": [after_missing_q],
    "filled": [before_missing_q - after_missing_q]
})

comparison

,metric,before,after,filled
0,SCOPUS Q missing,1524,445,1079


In [57]:
df_checkpoint2[COL_Q].value_counts(dropna=False)

SCOPUS Q
Q1      620
Q2      547
?       445
Q3      229
no-Q    196
Q4      116
Name: count, dtype: int64

In [58]:
output_file = "checkpoint_2_merged_scimago_database_q.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_checkpoint2.to_excel(writer, index=False, sheet_name="Data")

print(f"Saved: {output_file}")

Saved: checkpoint_2_merged_scimago_database_q.xlsx


check

In [59]:
# Data utama hasil checkpoint 2
q_missing_rows = df_checkpoint2[df_checkpoint2["SCOPUS Q"].apply(is_empty_value)].copy()

print("Jumlah baris SCOPUS Q masih ?/kosong:", len(q_missing_rows))

Jumlah baris SCOPUS Q masih ?/kosong: 445


In [60]:
q_missing_rows["journal_key"] = q_missing_rows["Journal"].apply(normalize_journal)
scimago_match_df["journal_key"] = scimago_match_df["journal"].apply(normalize_journal)

In [61]:
status_map = (
    scimago_match_df
    .drop_duplicates(subset=["journal_key"])
    .set_index("journal_key")["status"]
    .to_dict()
)

q_missing_rows["scimago_status"] = q_missing_rows["journal_key"].map(status_map)

In [62]:
q_missing_rows["scimago_status"].value_counts(dropna=False)

scimago_status
not_found     432
found_no_q     13
Name: count, dtype: int64